# Runoff contribution


In [ ]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import yaml 
import os

# plotting
from   plotly.subplots import make_subplots
import plotly.graph_objects as go

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes/')
os.chdir(wd)

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

In [ ]:
# Regions and Scenarios
regions = {
    'Outer tropics': (-20, -8),
    'Desert Andes': (-30, -20),
    'Central Andes': (-35, -30),
    'North Patagonia': (-45, -35),
    'South Patagonia': (-52, -45),
    'Fuegian Andes': (-56, -52)}

## Data

In [ ]:
# polygons / shapes
main_basins = gpd.read_file(cfg["dir"]["data"] + "/HYDROATLAS/hybas_sa_lev08_v1c_dissolved.shp")
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
hydroatlas_shape = hydroatlas_shape[hydroatlas_shape.glacier_area_RGI60 > 0]

# runoff in mm/day
hydroatlas_runoff_sim = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc").q_mm_sim

# runoff in m3/s 
hydroatlas_gmelt = xr.open_dataset("data/glacier/run_output_historical.nc").glacier_melt
hydroatlas_gmelt = hydroatlas_gmelt.reindex(gauge_id=hydroatlas_shape.index, fill_value=0)
hydroatlas_gmelt = (hydroatlas_gmelt*1000*86400) / (hydroatlas_shape.basin_area.values*1e6)[:, None]

# summer and annual averages
runoff_summer = hydroatlas_runoff_sim.sel(date=hydroatlas_runoff_sim.date.dt.month.isin([12, 1, 2]))  # filter for summer months
runoff_summer = runoff_summer.resample(date="YS").sum().mean(dim = "date")
runoff_summer = runoff_summer.rename("runoff_summer")
gmelt_summer = hydroatlas_gmelt.sel(date=hydroatlas_gmelt.date.dt.month.isin([12, 1, 2]))  # filter for summer months
gmelt_summer = gmelt_summer.resample(date="YS").sum().mean(dim = "date")
gmelt_summer = gmelt_summer.rename("gmelt_summer")

runoff_annual = hydroatlas_runoff_sim.resample(date="YS").sum().mean(dim = "date")
runoff_annual = runoff_annual.rename("runoff_annual")
gmelt_annual = hydroatlas_gmelt.resample(date="YS").sum().mean(dim = "date")
gmelt_annual = gmelt_annual.rename("gmelt_annual")

In [ ]:
hydroatlas_shape = hydroatlas_shape.join([runoff_annual.to_pandas(), 
                                          gmelt_annual.to_pandas(), 
                                          runoff_summer.to_pandas(), 
                                          gmelt_summer.to_pandas()])
hydroatlas_shape = hydroatlas_shape.dropna() # drop basins without glaciers
hydroatlas_shape["gmelt_prcp_ratio_annual"] = hydroatlas_shape["gmelt_annual"] / hydroatlas_shape["prcp_mean_ERA5"]
hydroatlas_shape["gmelt_runoff_ratio_annual"] = hydroatlas_shape["gmelt_summer"] / hydroatlas_shape["runoff_summer"]

hydroatlas_shape["lat_bands"] = pd.cut(hydroatlas_shape.basin_cenlat, 
                                       bins=range(-56, -7, 2), 
                                       labels=range(-56, -8, 2), 
                                       right=False)

## Plot

In [ ]:
# Create subplot structure
fig = make_subplots(
    rows=2, cols=2, shared_xaxes=True,
    horizontal_spacing=0.14, 
    vertical_spacing=0.03, 
    column_widths=[0.5, 0.5],
    specs=[[{"type": "xy", "rowspan": 2}, {"type": "xy"}],
           [None, {"type": "xy"}]]
)

# Define colorscale
colorscale = [(0, '#fde725'), (0.1, '#6ece58'), (0.2, '#35b779'), (0.6, '#31688e'), (1, '#440154')]

# Create box plots by latitudinal bands
fig.add_trace(
    go.Box(
        x=hydroatlas_shape['gmelt_runoff_ratio_annual']*100,
        y=hydroatlas_shape['lat_bands'],
        name="Basin runoff",
        marker_color='#31688e',
        fillcolor = '#31688e',
        orientation='h',
        marker_size=2,
        line_width=1,
        legendgroup="legend1",
    ), 
    row=1, col=1
)

fig.add_trace(
    go.Box(
        x=hydroatlas_shape['gmelt_prcp_ratio_annual']*100,
        y=hydroatlas_shape['lat_bands'],
        name="Precipitation",
        marker_color="#3fbd57",
        fillcolor = '#3fbd57',
        orientation='h',
        marker_size=2,
        line_width=1,
        legendgroup="legend1",
    ), 
    row=1, col=1
)

# Plot scatter plots
fig.add_trace(
    go.Scatter(
        x=hydroatlas_shape['gmelt_summer'],
        y=hydroatlas_shape['runoff_summer'],
        mode="markers",
        marker=dict(
            size = 8, 
            color=hydroatlas_shape['glacier_area_RGI60'],
            colorscale=colorscale,
            line =dict(width=0.5, color='darkgrey'),
            opacity = 1,
            showscale=True,
            colorbar=dict(title=dict(text="Glacier<br>area (%)", font=dict(size=15)), thickness = 15, 
                          ticklen=10, len = 0.92, y = 0.45)
        ),
        showlegend=False
    ),
    row=1, col=2
)

fig.add_trace(
    go.Scatter(
        x=hydroatlas_shape['gmelt_summer'],
        y=hydroatlas_shape['glacier_dhdt'] / 1000,
        mode="markers",
        marker=dict(
            size = 8, 
            color=hydroatlas_shape['glacier_area_RGI60'],
            line =dict(width=0.5, color='darkgrey'),
            colorscale=colorscale,
            showscale=False
        ),
        showlegend=False
    ),
    row=2, col=2
)

# Update axes
fig.update_xaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(ticks="outside", griddash="dot", zeroline=False)
fig.update_yaxes(title_standoff=2, ticksuffix="ºS", dtick = 5, range=(-58, -8), row=1, col=1)
fig.update_xaxes(title="Glacier melt ratio (%)", title_standoff = 2, type = "log", range = (-1,2.2), row=1, col=1)

fig.update_xaxes(title="Glacier melt (mm yr<sup>-1</sup>) [summer]", title_standoff = 0, range = (-100,3000), row=2, col=2)
fig.update_yaxes(title="Basin runoff (mm yr<sup>-1</sup>) [summer]", title_standoff = 2, range = (-100,3000), row=1, col=2)
fig.update_yaxes(title="Glacier dh/dt (m w.e. yr<sup>-1</sup>)", title_standoff = 8, ticksuffix=".0", row=2, col=2)


# add annotations 
fig.add_annotation(text="a)", xref="paper", yref="paper", x=0.40, y=1, showarrow=False, font=dict(size=16))
fig.add_annotation(text="b)", xref="paper", yref="paper", x=0.99, y=1, showarrow=False, font=dict(size=16))
fig.add_annotation(text="c)", xref="paper", yref="paper", x=0.99, y=0.46, showarrow=False, font=dict(size=16))

# Add vertical line on the left
fig.add_shape(
    type="line",
    x0=-0.12, x1=-0.12,
    y0=-0.02, y1=1.0,
    xref="paper", yref="paper",
    line=dict(color="black", width=1)
)

# Add horizontal lines at region boundaries
for boundary_lat in [-7, -20, -30, -35, -45, -52, -57]:
    y_pos = (boundary_lat - (-56)) / ((-7) - (-56))
    fig.add_shape(
        type="line",
        x0=-0.13, x1=-0.11,
        y0=y_pos, y1=y_pos,
        xref="paper", yref="paper",
        line=dict(color="black", width=1)
    )

# Add region annotations on the vertical line
for region, (lat_min, lat_max) in regions.items():
    # Calculate midpoint of the region in latitude

    mid_lat = (lat_min + lat_max) / 2
    y_pos = (mid_lat - (-56)) / ((-7) - (-56))
    
    fig.add_annotation(
        text=region.replace(' ', '<br>'),
        bgcolor="white", 
        x=-0.12,
        y=y_pos,
        xref="paper",
        yref="paper",
        showarrow=False,
        font=dict(size=13, family="Arial"),
        xanchor="center",
        yanchor="middle",
    )

# Update layout
fig.update_layout(
    autosize=False, 
    barmode='group',
    width=900, 
    height=600, 
    boxmode ='group',
    plot_bgcolor="rgba(229,236,246,0.7)",
    margin=dict(l=115, r=10, b=10, t=10), 
    font=dict(size=13, family="Arial"),
)

fig.write_image("figures/fig_runoff_contribution.svg", width=900, height=600)
fig.show()